# RED-diff / DPS FFHQ Single Run In Colab

This notebook is a thin Colab launcher for the adapted RED-diff repo in this workspace.

It uses the vendored `mycode2` measurement operators under `measurements/`, and it can auto-apply the RED-diff paper settings for the supported inverse problems.

It is designed around these operator-backed tasks in `main.py`:

- `down_sampling` for 4x super-resolution
- `gaussian_blur`
- `motion_blur`
- `inpainting_box`
- `inpainting_random`
- `phase_retrieval`

Each run writes a live `progress.json` file and a `metrics_history.json` file that you can inspect from Colab while sampling is still running.

Use `drive_zip` unless you have already pushed this adapted repo to your own fork. Cloning the upstream NVlabs repo directly will miss the operator and Colab patches added here.


## Runtime

In Colab, go to `Runtime > Change runtime type` and choose:

- Hardware accelerator: `GPU`
- Python version: default Colab runtime

The first run will auto-download the FFHQ checkpoint into the experiment root.


In [ ]:
#@title Project Settings

SETUP_MODE = "git"  #@param ["git", "drive_zip"]
REPO_URL = "https://github.com/Seif-Hussein/daps_test.git"  #@param {type:"string"}
REPO_BRANCH = "codex-reddiff-colab-operators"  #@param {type:"string"}
DRIVE_ZIP_PATH = "/content/drive/MyDrive/RED-diff-adapted.zip"  #@param {type:"string"}

REPO_DIR = "/content/RED-diff"  #@param {type:"string"}
PYTHON_BIN = "/usr/bin/python3"  #@param {type:"string"}
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/reddiff_single_run_exports"  #@param {type:"string"}
DRIVE_DATA_DIR = "/content/drive/MyDrive/mycode/test-ffhq"  #@param {type:"string"}
SESSION_TAG = ""  #@param {type:"string"}
RUN_NAME = "REDDiff_Single_Run"  #@param {type:"string"}
BASE_CONFIG_NAME = "ffhq256_uncond"  #@param ["ffhq256_uncond"]

ALGO_NAME = "reddiff"  #@param ["reddiff", "dps"]
INVERSE_TASK = "gaussian_blur"  #@param ["down_sampling", "gaussian_blur", "motion_blur", "inpainting_box", "inpainting_random", "phase_retrieval"]
USE_PAPER_REDDIFF_PRESET = True  #@param {type:"boolean"}

SEED = 42  #@param {type:"integer"}
TOTAL_IMAGES = 100  #@param {type:"integer"}
BATCH_SIZE = 10  #@param {type:"integer"}
DATA_START_IDX = 0  #@param {type:"integer"}
NUM_WORKERS = 2  #@param {type:"integer"}

# Manual fallback values. When USE_PAPER_REDDIFF_PRESET is enabled with ALGO_NAME="reddiff",
# the notebook replaces these with the paper setting for the task, or the closest reported task.
NUM_STEPS = 1000  #@param {type:"integer"}
MEASUREMENT_SIGMA = 0.0  #@param {type:"number"}
GRAD_TERM_WEIGHT = 0.25  #@param {type:"number"}
REDDIFF_LR = 0.5  #@param {type:"number"}
DPS_ETA = 0.5  #@param {type:"number"}

SCALE_FACTOR = 4  #@param {type:"integer"}
KERNEL_SIZE = 61  #@param {type:"integer"}
GAUSSIAN_INTENSITY = 3.0  #@param {type:"number"}
MOTION_INTENSITY = 0.5  #@param {type:"number"}
OVERSAMPLE = 2.0  #@param {type:"number"}
MASK_PROB_MIN = 0.70  #@param {type:"number"}
MASK_PROB_MAX = 0.71  #@param {type:"number"}

# Used only for inpainting_box. These are passed explicitly so the masked pixels stay fixed.
BOX_TOP = 64  #@param {type:"integer"}
BOX_LEFT = 64  #@param {type:"integer"}
BOX_HEIGHT = 128  #@param {type:"integer"}
BOX_WIDTH = 128  #@param {type:"integer"}

SAVE_ORI = False  #@param {type:"boolean"}
SAVE_DEG = False  #@param {type:"boolean"}
SAVE_EVOLUTION = False  #@param {type:"boolean"}
SMOKE_TEST = 0  #@param {type:"integer"}
LOG_TAIL_LINES = 120  #@param {type:"integer"}
PREVIEW_LIMIT = 6  #@param {type:"integer"}

# Optional extra Hydra overrides, separated by semicolons.
EXTRA_HYDRA_OVERRIDES = ""  #@param {type:"string"}


In [ ]:
#@title Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
#@title Fetch The Repo
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
repo_dir.parent.mkdir(parents=True, exist_ok=True)
os.chdir(repo_dir.parent)

if repo_dir.exists():
    shutil.rmtree(repo_dir)

if SETUP_MODE == "git":
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            REPO_URL,
            repo_dir.as_posix(),
        ],
        check=True,
    )
elif SETUP_MODE == "drive_zip":
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(repo_dir.parent)
    extracted_root = repo_dir.parent / zip_path.stem
    if extracted_root.exists() and extracted_root != repo_dir:
        if repo_dir.exists():
            shutil.rmtree(repo_dir)
        extracted_root.rename(repo_dir)
else:
    raise ValueError(f"Unsupported SETUP_MODE: {SETUP_MODE}")

os.chdir(repo_dir)
print(f"Repo ready: {repo_dir}")


In [ ]:
#@title Install Colab Dependencies
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
req_path = Path("requirements-colab.txt")
if req_path.exists():
    subprocess.run([PYTHON_BIN, "-m", "pip", "install", "-q", "-r", req_path.as_posix()], check=True)
    print(f"Installed {req_path}")
else:
    fallback = [
        "hydra-core==1.3.2",
        "omegaconf==2.3.0",
        "gdown==5.2.0",
        "lmdb>=1.4,<2",
        "tensorboard>=2.14",
        "scipy==1.14.1",
        "torchmetrics>=1.4,<2",
    ]
    subprocess.run([PYTHON_BIN, "-m", "pip", "install", "-q", *fallback], check=True)
    print("Installed fallback Colab packages")


In [ ]:
#@title Build Single-Run Command
import json
import os
import shlex
import time
from pathlib import Path

os.chdir(REPO_DIR)
repo_dir = Path(REPO_DIR)
drive_data_dir = Path(DRIVE_DATA_DIR)
if not drive_data_dir.exists():
    raise FileNotFoundError(f"Dataset path not found: {drive_data_dir}")

IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".pgm", ".tif", ".tiff", ".webp"}

PAPER_REDDIFF_PRESETS = {
    "down_sampling": {
        "paper_task": "4x superresolution",
        "closest_task": "4x superresolution",
        "exact_match": True,
        "num_steps": 1000,
        "measurement_sigma": 0.0,
        "grad_term_weight": 0.25,
        "lr": 0.25,
    },
    "inpainting_random": {
        "paper_task": "inpainting",
        "closest_task": "inpainting",
        "exact_match": True,
        "num_steps": 1000,
        "measurement_sigma": 0.0,
        "grad_term_weight": 0.25,
        "lr": 0.5,
    },
    "inpainting_box": {
        "paper_task": None,
        "closest_task": "inpainting",
        "exact_match": False,
        "num_steps": 1000,
        "measurement_sigma": 0.0,
        "grad_term_weight": 0.25,
        "lr": 0.5,
    },
    "phase_retrieval": {
        "paper_task": "phase retrieval",
        "closest_task": "phase retrieval",
        "exact_match": True,
        "num_steps": 1000,
        "measurement_sigma": 0.0,
        "grad_term_weight": 0.25,
        "lr": 0.5,
    },
    "gaussian_blur": {
        "paper_task": None,
        "closest_task": "deblurring",
        "exact_match": False,
        "num_steps": 1000,
        "measurement_sigma": 0.0,
        "grad_term_weight": 0.25,
        "lr": 0.5,
    },
    "motion_blur": {
        "paper_task": None,
        "closest_task": "deblurring",
        "exact_match": False,
        "num_steps": 1000,
        "measurement_sigma": 0.0,
        "grad_term_weight": 0.25,
        "lr": 0.5,
    },
}

def parse_list(text: str):
    items = []
    for raw_item in text.replace("\n", ";").split(";"):
        raw_item = raw_item.strip()
        if raw_item:
            items.append(raw_item)
    return items

def collect_relpaths(root: Path):
    relpaths = []
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMG_EXTENSIONS:
            relpaths.append(path.relative_to(root).as_posix())
    return sorted(relpaths)

paper_preset = None
effective_num_steps = int(NUM_STEPS)
effective_measurement_sigma = float(MEASUREMENT_SIGMA)
effective_grad_term_weight = float(GRAD_TERM_WEIGHT)
effective_reddiff_lr = float(REDDIFF_LR)
effective_denoise_term_weight = "linear"

if ALGO_NAME == "reddiff" and USE_PAPER_REDDIFF_PRESET:
    paper_preset = dict(PAPER_REDDIFF_PRESETS[INVERSE_TASK])
    effective_num_steps = int(paper_preset["num_steps"])
    effective_measurement_sigma = float(paper_preset["measurement_sigma"])
    effective_grad_term_weight = float(paper_preset["grad_term_weight"])
    effective_reddiff_lr = float(paper_preset["lr"])

extra_overrides = parse_list(EXTRA_HYDRA_OVERRIDES)
session_tag = SESSION_TAG.strip() or time.strftime("%Y%m%d-%H%M%S")
run_slug = f"{ALGO_NAME}_{INVERSE_TASK}_{session_tag}"
run_name = f"{RUN_NAME}_{session_tag}"

all_relpaths = collect_relpaths(drive_data_dir)
if not all_relpaths:
    raise FileNotFoundError(f"No images found under {drive_data_dir}")

start_idx = int(DATA_START_IDX)
end_idx = min(start_idx + int(TOTAL_IMAGES), len(all_relpaths))
if start_idx >= len(all_relpaths):
    raise ValueError(f"DATA_START_IDX={start_idx} is outside the dataset of size {len(all_relpaths)}")

selected_relpaths = all_relpaths[start_idx:end_idx]
effective_total_images = len(selected_relpaths)
effective_batch_size = min(int(BATCH_SIZE), effective_total_images)

run_aux_root = repo_dir / "colab_runs"
subset_root = run_aux_root / "subsets"
subset_root.mkdir(parents=True, exist_ok=True)
subset_path = subset_root / f"{run_slug}.txt"
subset_path.write_text("".join(f"{relpath} 0\n" for relpath in selected_relpaths), encoding="utf-8")

exp_root = run_aux_root
samples_root = exp_root / "samples" / run_slug
progress_path = samples_root / "progress.json"
metrics_history_path = samples_root / "metrics_history.json"
hydra_root = repo_dir / "outputs" / "colab_runs" / run_slug
latest_log_path = run_aux_root / f"{run_slug}.log"
latest_pid_path = run_aux_root / f"{run_slug}.pid"

run_cmd = [
    PYTHON_BIN,
    "main.py",
    "-cn",
    BASE_CONFIG_NAME,
    "dataset=drive_images_256",
    f"dataset.root={drive_data_dir.as_posix()}",
    f"dataset.subset_txt={subset_path.as_posix()}",
    f"algo={ALGO_NAME}",
    f"algo.deg={INVERSE_TASK}",
    "classifier=none",
    "dist.num_processes_per_node=1",
    f"exp.root={exp_root.as_posix()}",
    f"exp.name={run_slug}",
    "exp.samples_root=samples",
    "exp.overwrite=true",
    f"exp.seed={SEED}",
    f"exp.num_steps={effective_num_steps}",
    "exp.write_progress_json=true",
    "exp.progress_json=progress.json",
    "exp.write_metrics_history_json=true",
    "exp.metrics_history_json=metrics_history.json",
    f"exp.save_ori={'true' if SAVE_ORI else 'false'}",
    f"exp.save_deg={'true' if SAVE_DEG else 'false'}",
    f"exp.save_evolution={'true' if SAVE_EVOLUTION else 'false'}",
    f"exp.smoke_test={SMOKE_TEST}",
    f"loader.batch_size={effective_batch_size}",
    f"loader.num_workers={NUM_WORKERS}",
    "loader.shuffle=false",
    "loader.drop_last=false",
    "loader.pin_memory=true",
    f"algo.sigma_y={effective_measurement_sigma}",
    f"algo.operator.sigma={effective_measurement_sigma}",
    f"algo.grad_term_weight={effective_grad_term_weight}",
]

if ALGO_NAME == "reddiff":
    run_cmd.extend([
        "algo.awd=true",
        f"algo.lr={effective_reddiff_lr}",
        f"algo.denoise_term_weight={effective_denoise_term_weight}",
    ])
elif ALGO_NAME == "dps":
    run_cmd.extend([
        "algo.awd=true",
        f"algo.eta={DPS_ETA}",
    ])

if INVERSE_TASK == "down_sampling":
    run_cmd.append(f"algo.operator.scale_factor={SCALE_FACTOR}")
elif INVERSE_TASK == "gaussian_blur":
    run_cmd.extend([
        f"algo.operator.kernel_size={KERNEL_SIZE}",
        f"algo.operator.gaussian_intensity={GAUSSIAN_INTENSITY}",
    ])
elif INVERSE_TASK == "motion_blur":
    run_cmd.extend([
        f"algo.operator.kernel_size={KERNEL_SIZE}",
        f"algo.operator.motion_intensity={MOTION_INTENSITY}",
    ])
elif INVERSE_TASK == "phase_retrieval":
    run_cmd.append(f"algo.operator.oversample={OVERSAMPLE}")
elif INVERSE_TASK == "inpainting_random":
    run_cmd.extend([
        f"algo.operator.mask_prob_range=[{MASK_PROB_MIN},{MASK_PROB_MAX}]",
    ])
elif INVERSE_TASK == "inpainting_box":
    run_cmd.extend([
        f"algo.operator.box_start=[{BOX_TOP},{BOX_LEFT}]",
        f"algo.operator.box_mask_len=[{BOX_HEIGHT},{BOX_WIDTH}]",
    ])

run_cmd.extend(extra_overrides)

print(f"Run slug: {run_slug}")
print(f"Run name: {run_name}")
print(f"Config: {BASE_CONFIG_NAME}")
print(f"Algorithm: {ALGO_NAME}")
print(f"Task: {INVERSE_TASK}")
print(f"Dataset slice: [{start_idx}, {end_idx}) out of {len(all_relpaths)} images")
print(f"Effective batch size: {effective_batch_size}")
print(f"Progress JSON: {progress_path}")
print(f"Metrics history JSON: {metrics_history_path}")
print(f"Subset file: {subset_path}")
print(f"Samples root: {samples_root}")
print(f"Hydra root: {hydra_root}")
print("Measurement operators: vendored from mycode2 under measurements/")
if paper_preset is not None:
    if paper_preset["exact_match"]:
        print(f"RED-diff paper preset: exact match for {paper_preset['paper_task']}")
    else:
        print(f"RED-diff paper preset: closest reported task is {paper_preset['closest_task']}")
    print(json.dumps(paper_preset, indent=2))
if extra_overrides:
    print(f"Extra overrides: {extra_overrides}")

print("\nCommand:\n")
print(" ".join(shlex.quote(part) for part in run_cmd))

last_context = {
    "run_slug": run_slug,
    "run_name": run_name,
    "samples_root": samples_root.as_posix(),
    "progress_path": progress_path.as_posix(),
    "metrics_history_path": metrics_history_path.as_posix(),
    "hydra_root": hydra_root.as_posix(),
    "subset_path": subset_path.as_posix(),
    "latest_log_path": latest_log_path.as_posix(),
    "latest_pid_path": latest_pid_path.as_posix(),
    "measurement_operator_source": "vendored mycode2 operators in measurements/",
    "paper_reddiff_preset": paper_preset,
    "run_cmd": run_cmd,
}

context_path = run_aux_root / f"{run_slug}.context.json"
last_context["context_path"] = context_path.as_posix()
context_path.write_text(json.dumps(last_context, indent=2), encoding="utf-8")
print(f"Context saved to: {context_path}")


In [ ]:
#@title Launch The Run In Background
import os
import subprocess
from pathlib import Path

os.chdir(REPO_DIR)
latest_log_path = Path(last_context["latest_log_path"])
latest_pid_path = Path(last_context["latest_pid_path"])
latest_log_path.parent.mkdir(parents=True, exist_ok=True)

launch_env = os.environ.copy()
launch_env["PYTHONUNBUFFERED"] = "1"

with latest_log_path.open("w", encoding="utf-8") as log_handle:
    process = subprocess.Popen(
        last_context["run_cmd"],
        cwd=REPO_DIR,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
        env=launch_env,
    )

latest_pid_path.write_text(str(process.pid), encoding="utf-8")
print(f"PID: {process.pid}")
print(f"Log: {latest_log_path}")
print(f"Samples root: {last_context['samples_root']}")


In [ ]:
#@title Show Status, Recent Logs, And Sample Previews
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

log_path = Path(last_context["latest_log_path"])
pid_path = Path(last_context["latest_pid_path"])
samples_root = Path(last_context["samples_root"])
progress_path = Path(last_context["progress_path"])
metrics_history_path = Path(last_context["metrics_history_path"])

alive = False
pid = None
if pid_path.exists():
    pid = int(pid_path.read_text(encoding="utf-8").strip())
    try:
        os.kill(pid, 0)
        alive = True
    except OSError:
        alive = False

print(f"PID: {pid}")
print(f"Running: {alive}")
print(f"Log path: {log_path}")
print(f"Progress JSON: {progress_path}")
print(f"Metrics history JSON: {metrics_history_path}")
print(f"Samples root: {samples_root}")

if progress_path.exists():
    progress = json.loads(progress_path.read_text(encoding="utf-8"))
    print("\nProgress summary:\n")
    print(f"status: {progress.get('status')}")
    print(f"completed_batches: {progress.get('completed_batches')} / {progress.get('planned_batches')}")
    print(f"completed_images: {progress.get('completed_images')} / {progress.get('planned_images')}")
    print(f"elapsed_sec: {progress.get('elapsed_sec')}")
    print(f"eta_sec: {progress.get('eta_sec')}")
    print(f"mean_psnr: {progress.get('mean_psnr')}")
    print(f"mean_ssim: {progress.get('mean_ssim')}")
    print(f"mean_lpips: {progress.get('mean_lpips')}")
    if progress.get('error'):
        print(f"error: {progress['error']}")
    print("\nRaw progress JSON:\n")
    print(json.dumps(progress, indent=2))
else:
    print("\nProgress JSON does not exist yet.")

if metrics_history_path.exists():
    metrics_history = json.loads(metrics_history_path.read_text(encoding="utf-8"))
    entries = metrics_history.get("entries", [])
    print(f"\nMetric history entries: {len(entries)}")
    if entries:
        latest = entries[-1]
        print("Latest metric history entry:\n")
        print(json.dumps(latest, indent=2))
else:
    print("\nMetrics history JSON does not exist yet.")

if log_path.exists():
    lines = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    tail = lines[-int(LOG_TAIL_LINES):]
    print("\nRecent log lines:\n")
    print("\n".join(tail) if tail else "<log is empty>")
else:
    print("\nLog file does not exist yet.")

sample_paths = sorted(samples_root.rglob("*.png")) if samples_root.exists() else []
print(f"\nSaved PNGs: {len(sample_paths)}")
for path in sample_paths[:10]:
    print(path.as_posix())

preview_paths = sample_paths[:int(PREVIEW_LIMIT)]
if preview_paths:
    cols = min(3, len(preview_paths))
    rows = (len(preview_paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for ax, image_path in zip(axes, preview_paths):
        ax.imshow(Image.open(image_path))
        ax.set_title(image_path.name)
        ax.axis("off")
    for ax in axes[len(preview_paths):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("\nNo preview images found yet.")


In [ ]:
#@title Copy Run Artifacts To Drive
import shutil
from pathlib import Path

export_root = Path(DRIVE_EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)

targets = [
    Path(last_context["samples_root"]),
    Path(last_context["progress_path"]),
    Path(last_context["metrics_history_path"]),
    Path(last_context["hydra_root"]),
    Path(last_context["latest_log_path"]),
    Path(last_context["subset_path"]),
    Path(last_context["context_path"]),
]

for src in targets:
    if not src.exists():
        print(f"Skipping missing path: {src}")
        continue

    dst = export_root / src.name
    if src.is_dir():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst}")
